## Step 1: Load Master Dataset

In [1]:
import pandas as pd
import numpy as np
import pickle
import os

# Output folder structure — keeps intermediate state, final numbers,
# and visuals separated so a partial run never gets lost or overwritten
for folder in ["results", "artifacts"]:
    os.makedirs(folder, exist_ok=True)

# Load the master dataset
df = pd.read_csv("master_dataset.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)


Shape: (6175277, 56)

Columns:
['Id_x', 'PATIENT', 'ORGANIZATION', 'VISIT_DATE', 'REPORTED_DATE', 'VISIT_YEAR', 'VISIT_MONTH', 'VISIT_WEEK', 'VISIT_DAY', 'ENCOUNTERCLASS', 'Id_y', 'BIRTHDATE', 'DEATHDATE', 'GENDER', 'CITY', 'STATE', 'COUNTRY', 'LAT', 'LON', 'SEASON', 'COMORBIDITY_COUNT', 'AGE_AT_VISIT', 'DISEASE', 'VISIT_NUMBER', 'sym_fever', 'sym_chills', 'sym_headache', 'sym_fatigue', 'sym_vomiting', 'sym_diarrhea', 'sym_cough', 'sym_shortness_of_breath', 'sym_chest_pain', 'sym_rash', 'sym_muscle_pain', 'sym_abdominal_pain', 'sym_night_sweats', 'sym_weight_loss', 'sym_jaundice', 'sym_stiff_neck', 'sym_sensitivity_to_light', 'sym_red_eyes', 'sym_runny_nose', 'sym_sore_throat', 'sym_hemorrhage', 'sym_sweating', 'sym_dehydration', 'sym_muscle_cramps', 'sym_loss_of_smell', 'sym_swollen_lymph_nodes', 'sym_pain_behind_eyes', 'sym_general_weakness', 'IS_OUTBREAK', 'IS_BACKGROUND_NOISE', 'IS_SECONDARY_SPREAD', 'HOSPITAL_NAME']

First 3 rows:


,Id_x,PATIENT,ORGANIZATION,VISIT_DATE,REPORTED_DATE,VISIT_YEAR,VISIT_MONTH,VISIT_WEEK,VISIT_DAY,ENCOUNTERCLASS,...,sym_dehydration,sym_muscle_cramps,sym_loss_of_smell,sym_swollen_lymph_nodes,sym_pain_behind_eyes,sym_general_weakness,IS_OUTBREAK,IS_BACKGROUND_NOISE,IS_SECONDARY_SPREAD,HOSPITAL_NAME
0,8fca053a-7c68-87b2-8825-81202de60368,8fca053a-7c68-87b2-a1bf-7fe1774b6586,Hospital_Fayyum,2000-01-01 00:16:23,2000-01-01 00:16:23,2000,1,52,1,ambulatory,...,0,0,0,0,0,0,0,0,0,Hospital_Fayyum
1,2250a98d-4a3e-8b57-b198-4eabce30ea46,2250a98d-4a3e-8b57-058b-96fb10239398,Hospital_Ismailia,2000-01-01 00:27:01,2000-01-01 00:27:01,2000,1,52,1,ambulatory,...,0,0,0,0,0,0,0,0,0,Hospital_Ismailia
2,fcd12547-4425-dabe-44fa-75323dbea7f3,fcd12547-4425-dabe-0900-bb75a13a21fa,Hospital_Shibin El Kom,2000-01-01 00:37:48,2000-01-01 00:37:48,2000,1,52,1,ambulatory,...,0,0,0,0,0,0,0,0,0,Hospital_Shibin El Kom


## Step 2: Drop Leakage / Unused Columns

In [2]:
# Columns to drop (data leakage or not useful for training)
columns_to_drop = [
    "IS_BACKGROUND_NOISE",
    "IS_SECONDARY_SPREAD",
    "REPORTED_DATE",
    "Id_x",
    "Id_y",
    "PATIENT",
    "ORGANIZATION",
    "BIRTHDATE",
    "DEATHDATE",
]

df = df.drop(columns=columns_to_drop, errors="ignore")

print("Shape after dropping columns:", df.shape)
print(df.columns.tolist())

Shape after dropping columns: (6175277, 47)
['VISIT_DATE', 'VISIT_YEAR', 'VISIT_MONTH', 'VISIT_WEEK', 'VISIT_DAY', 'ENCOUNTERCLASS', 'GENDER', 'CITY', 'STATE', 'COUNTRY', 'LAT', 'LON', 'SEASON', 'COMORBIDITY_COUNT', 'AGE_AT_VISIT', 'DISEASE', 'VISIT_NUMBER', 'sym_fever', 'sym_chills', 'sym_headache', 'sym_fatigue', 'sym_vomiting', 'sym_diarrhea', 'sym_cough', 'sym_shortness_of_breath', 'sym_chest_pain', 'sym_rash', 'sym_muscle_pain', 'sym_abdominal_pain', 'sym_night_sweats', 'sym_weight_loss', 'sym_jaundice', 'sym_stiff_neck', 'sym_sensitivity_to_light', 'sym_red_eyes', 'sym_runny_nose', 'sym_sore_throat', 'sym_hemorrhage', 'sym_sweating', 'sym_dehydration', 'sym_muscle_cramps', 'sym_loss_of_smell', 'sym_swollen_lymph_nodes', 'sym_pain_behind_eyes', 'sym_general_weakness', 'IS_OUTBREAK', 'HOSPITAL_NAME']


## Step 3: Encode Categorical Columns
Fitted on the full dataset — this is fine for label encoding since the categorical vocabulary (cities, diseases, seasons...) is fixed and known in advance; it is not target leakage. It is saved immediately so it can be reused without refitting.

In [3]:
from sklearn.preprocessing import LabelEncoder

# Categorical columns to encode
categorical_cols = ["CITY", "COUNTRY", "DISEASE", "SEASON", "GENDER"]

encoders = {}  # store encoders for later use (inference / reverse mapping)

for col in categorical_cols:
    le = LabelEncoder()
    df[col + "_ENC"] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

print(df[[c for c in categorical_cols] + [c + "_ENC" for c in categorical_cols]].head())
print("\nNumber of unique values per column:")
for col in categorical_cols:
    print(col, "->", df[col].nunique())

            CITY       COUNTRY      DISEASE      SEASON GENDER  CITY_ENC  \
0         Fayyum         Egypt  Hepatitis A    Cool_Dry      M        22   
1       Ismailia         Egypt    Influenza    Cool_Dry      M        30   
2  Shibin El Kom         Egypt    Influenza    Cool_Dry      F        56   
3   Johannesburg  South Africa     HIV/AIDS  Dry_Season      F        31   
4        Kampala        Uganda    Influenza    Cool_Dry      F        34   

   COUNTRY_ENC  DISEASE_ENC  SEASON_ENC  GENDER_ENC  
0            3            6           0           1  
1            3            7           0           1  
2            3            7           0           0  
3           12            5           1           0  
4           16            7           0           0  

Number of unique values per column:
CITY -> 64
COUNTRY -> 19
DISEASE -> 18
SEASON -> 4
GENDER -> 2


In [4]:
# Checkpoint: save encoders right away so this step never needs to be rerun
with open("artifacts/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

print("Saved artifacts/encoders.pkl")


Saved artifacts/encoders.pkl


## Step 4: Chronological Train / Validation / Test Split
Split **before** scaling, so the scaler never sees validation/test data.

In [5]:
# Sort by VISIT_DATE to ensure correct chronological order
df = df.sort_values("VISIT_DATE").reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

# Sanity check: sizes
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# Sanity check: date ranges (must not overlap)
print("\nTrain date range:", train_df["VISIT_DATE"].min(), "->", train_df["VISIT_DATE"].max())
print("Validation date range:", val_df["VISIT_DATE"].min(), "->", val_df["VISIT_DATE"].max())
print("Test date range:", test_df["VISIT_DATE"].min(), "->", test_df["VISIT_DATE"].max())

# Sanity check: IS_OUTBREAK balance across splits
print("\nOutbreak ratio per split:")
print("Train:", train_df["IS_OUTBREAK"].mean().round(4))
print("Validation:", val_df["IS_OUTBREAK"].mean().round(4))
print("Test:", test_df["IS_OUTBREAK"].mean().round(4))

Train: (4322693, 52)
Validation: (926292, 52)
Test: (926292, 52)

Train date range: 2000-01-01 00:16:23 -> 2022-09-11 16:10:00
Validation date range: 2022-09-11 16:10:28 -> 2024-08-15 18:01:33
Test date range: 2024-08-15 18:06:28 -> 2026-07-10 10:00:13

Outbreak ratio per split:
Train: 0.0545
Validation: 0.0275
Test: 0.0274


## Step 5: Scale Numerical Columns — Fit on Train Only
The scaler is fit on `train_df` only, then used to `transform` (never `fit`) validation and test, so no distributional information leaks across splits.

In [6]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ["AGE_AT_VISIT", "LAT", "LON", "COMORBIDITY_COUNT"]

scaler = StandardScaler()
scaler.fit(train_df[numerical_cols])  # fit on TRAIN ONLY

for split_df in (train_df, val_df, test_df):
    split_df[[c + "_NORM" for c in numerical_cols]] = scaler.transform(split_df[numerical_cols])

# Sanity check
print("Train stats:")
print(train_df[numerical_cols + [c + "_NORM" for c in numerical_cols]].describe())

Train stats:
       AGE_AT_VISIT           LAT           LON  COMORBIDITY_COUNT  \
count  4.322693e+06  4.322693e+06  4.322693e+06       4.322693e+06   
mean   4.642615e+01  1.248529e+01  2.264962e+01       2.687655e+00   
std    2.434218e+01  1.839137e+01  1.485613e+01       2.626990e+00   
min    8.000000e-02 -3.392490e+01 -1.746770e+01       0.000000e+00   
25%    2.655989e+01  3.476000e-01  8.592000e+00       1.000000e+00   
50%    4.860000e+01  1.200220e+01  3.100040e+01       2.000000e+00   
75%    6.418070e+01  3.004440e+01  3.247770e+01       3.000000e+00   
max    1.000000e+02  3.681900e+01  4.186610e+01       1.000000e+01   

       AGE_AT_VISIT_NORM      LAT_NORM      LON_NORM  COMORBIDITY_COUNT_NORM  
count       4.322693e+06  4.322693e+06  4.322693e+06            4.322693e+06  
mean        1.062862e-15  8.354459e-16 -5.321082e-16            1.035168e-16  
std         1.000000e+00  1.000000e+00  1.000000e+00            1.000000e+00  
min        -1.903945e+00 -2.523476e+00 -

In [7]:
# Checkpoint: save the scaler right away so this step never needs to be rerun
with open("artifacts/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Saved artifacts/scaler.pkl")


Saved artifacts/scaler.pkl


## Step 6: Save Preprocessed Splits
Save train/val/test as CSVs. If the kernel or machine stops here, just reload these three files plus `scaler.pkl` and `encoders.pkl` — no need to rerun anything above.

In [8]:
train_df.to_csv("artifacts/train_data.csv", index=False)
val_df.to_csv("artifacts/val_data.csv", index=False)
test_df.to_csv("artifacts/test_data.csv", index=False)

print("Saved artifacts/train_data.csv  :", train_df.shape)
print("Saved artifacts/val_data.csv    :", val_df.shape)
print("Saved artifacts/test_data.csv   :", test_df.shape)
print("\nAll checkpoint files ready in artifacts/: train_data.csv, val_data.csv, "
      "test_data.csv, scaler.pkl, encoders.pkl")


Saved artifacts/train_data.csv  : (4322693, 56)
Saved artifacts/val_data.csv    : (926292, 56)
Saved artifacts/test_data.csv   : (926292, 56)

All checkpoint files ready in artifacts/: train_data.csv, val_data.csv, test_data.csv, scaler.pkl, encoders.pkl


## Resume Cell — Run This Instead of Everything Above
If the session was interrupted after Step 6 completed once, skip Steps 1-6 entirely and just run this cell to reload everything needed for training.

In [9]:
import pandas as pd
import pickle

train_df = pd.read_csv("artifacts/train_data.csv")
val_df   = pd.read_csv("artifacts/val_data.csv")
test_df  = pd.read_csv("artifacts/test_data.csv")

with open("artifacts/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
with open("artifacts/encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

print("Reloaded:")
print("Train:", train_df.shape, "| Val:", val_df.shape, "| Test:", test_df.shape)
print("Scaler numerical cols:", list(scaler.feature_names_in_))
print("Encoders:", list(encoders.keys()))


Reloaded:
Train: (4322693, 56) | Val: (926292, 56) | Test: (926292, 56)
Scaler numerical cols: ['AGE_AT_VISIT', 'LAT', 'LON', 'COMORBIDITY_COUNT']
Encoders: ['CITY', 'COUNTRY', 'DISEASE', 'SEASON', 'GENDER']


---

# Task 3 — Anomaly Detection: DBSCAN + Isolation Forest

**Objective:** Detect epidemic outbreak patterns using unsupervised anomaly detection.

**Design rationale:** LAT/LON coordinates in this dataset are identical for all patients
in the same city (facility-level reporting, consistent with real systems like DHIS2).
Geographic DBSCAN on individual coordinates is therefore uninformative.
Instead, we apply city-chunked DBSCAN on symptom profiles and continuous time,
detecting *dense symptom-time clusters within each city-disease combination* as outbreak signals.
This matches the original proposal description: AI Symptom Clustering.

**Input:** `train_data.csv`, `val_data.csv`, `test_data.csv` (from Task 1)
**Models:** DBSCAN (symptom-temporal clustering) + Isolation Forest (baseline)
**Target:** `IS_OUTBREAK` (binary)


In [10]:
# Load preprocessed splits — no dependency on Task 1 cells above
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

for folder in ["results", "artifacts"]:
    os.makedirs(folder, exist_ok=True)

train_df = pd.read_csv("artifacts/train_data.csv")
val_df   = pd.read_csv("artifacts/val_data.csv")
test_df  = pd.read_csv("artifacts/test_data.csv")

print("Train :", train_df.shape)
print("Val   :", val_df.shape)
print("Test  :", test_df.shape)
print(f"Outbreak rate — Train: {train_df['IS_OUTBREAK'].mean():.4f} | "
      f"Val: {val_df['IS_OUTBREAK'].mean():.4f} | Test: {test_df['IS_OUTBREAK'].mean():.4f}")


Train : (4322693, 56)
Val   : (926292, 56)
Test  : (926292, 56)
Outbreak rate — Train: 0.0545 | Val: 0.0275 | Test: 0.0274


## Step 7: Feature Engineering for Anomaly Detection

In [11]:
# Continuous time feature: days from dataset start
# Avoids the cyclical discontinuity of raw VISIT_WEEK/VISIT_MONTH
# (week 52 and week 1 are adjacent but numerically far apart)
for df_ in (train_df, val_df, test_df):
    df_["VISIT_DATE"] = pd.to_datetime(df_["VISIT_DATE"])

t0 = train_df["VISIT_DATE"].min()
for df_ in (train_df, val_df, test_df):
    df_["DAY_INDEX"] = (df_["VISIT_DATE"] - t0).dt.days.astype(float)
    # Normalize to [0, 1] using training range
    day_max = train_df["DAY_INDEX"].max()
    df_["DAY_INDEX_NORM"] = df_["DAY_INDEX"] / day_max

SYM_COLS = [c for c in train_df.columns if c.startswith("sym_")]
print(f"Symptom columns: {len(SYM_COLS)}")


Symptom columns: 28


## Step 7b: Recent Case-Rate Feature — Collective Outbreak Signal

Point-level features (symptoms + date) cannot separate an outbreak case
from a background case with identical symptoms — the discriminating
signal is **collective**, not individual: "how many similar cases showed
up nearby in time in the same city and disease". This is exactly the
signal validated during exploration (city+disease burstiness gave
validation ROC-AUC ≈ 0.91 on its own).

This step adds `RECENT_CASE_COUNT`: the number of prior visits in the
same `CITY` + `DISEASE` within a trailing 7-day window.

- **Fully causal** — only counts visits strictly *before* the current
  one, so it never looks into the future.
- **No label leakage** — never touches `IS_OUTBREAK`, only visit dates.
- Computed on the full chronological timeline (train+val+test combined
  by date, then split back apart) so rows near a split boundary still
  get a correct trailing count — this uses only report timestamps that
  would be available in production regardless of the train/val/test cut,
  not the label.

In [12]:
CASE_RATE_WINDOW_DAYS = 7

def add_recent_case_rate(train_df, val_df, test_df, window_days=CASE_RATE_WINDOW_DAYS):
    """
    Adds RECENT_CASE_COUNT: number of prior visits in the same CITY+DISEASE
    within a trailing `window_days` window, computed causally (two-pointer
    per city-disease group, sorted by date). No IS_OUTBREAK is used.
    """
    train_df = train_df.copy(); train_df["_SPLIT"] = "train"
    val_df   = val_df.copy();   val_df["_SPLIT"]   = "val"
    test_df  = test_df.copy();  test_df["_SPLIT"]  = "test"

    full = pd.concat([train_df, val_df, test_df], ignore_index=True)
    full["VISIT_DATE"] = pd.to_datetime(full["VISIT_DATE"])
    full = full.sort_values("VISIT_DATE").reset_index(drop=True)

    counts = np.zeros(len(full))
    window_np = np.timedelta64(window_days, "D")

    for (city, disease), grp in full.groupby(["CITY", "DISEASE"], sort=False):
        dates = grp["VISIT_DATE"].values
        idx   = grp.index.values
        left  = 0
        for right in range(len(dates)):
            while dates[right] - dates[left] > window_np:
                left += 1
            counts[idx[right]] = right - left  # prior cases in window, excludes current row

    full["RECENT_CASE_COUNT"] = counts

    train_out = full[full["_SPLIT"] == "train"].drop(columns=["_SPLIT"]).reset_index(drop=True)
    val_out   = full[full["_SPLIT"] == "val"].drop(columns=["_SPLIT"]).reset_index(drop=True)
    test_out  = full[full["_SPLIT"] == "test"].drop(columns=["_SPLIT"]).reset_index(drop=True)
    return train_out, val_out, test_out


train_df, val_df, test_df = add_recent_case_rate(train_df, val_df, test_df)

# Scale using TRAIN stats only — consistent with the rest of the pipeline
from sklearn.preprocessing import StandardScaler
case_rate_scaler = StandardScaler()
train_df["RECENT_CASE_COUNT_NORM"] = case_rate_scaler.fit_transform(train_df[["RECENT_CASE_COUNT"]])
val_df["RECENT_CASE_COUNT_NORM"]   = case_rate_scaler.transform(val_df[["RECENT_CASE_COUNT"]])
test_df["RECENT_CASE_COUNT_NORM"]  = case_rate_scaler.transform(test_df[["RECENT_CASE_COUNT"]])

with open("artifacts/case_rate_scaler.pkl", "wb") as f:
    pickle.dump(case_rate_scaler, f)

print("Saved artifacts/case_rate_scaler.pkl")
print()
print(f"Mean RECENT_CASE_COUNT — outbreak rows   : "
      f"{train_df.loc[train_df.IS_OUTBREAK==1, 'RECENT_CASE_COUNT'].mean():.2f}")
print(f"Mean RECENT_CASE_COUNT — background rows : "
      f"{train_df.loc[train_df.IS_OUTBREAK==0, 'RECENT_CASE_COUNT'].mean():.2f}")


Saved artifacts/case_rate_scaler.pkl

Mean RECENT_CASE_COUNT — outbreak rows   : 291.57
Mean RECENT_CASE_COUNT — background rows : 36.51


In [13]:
# Isolation Forest uses the raw feature set directly.
# DBSCAN uses a PCA-reduced version of the symptom columns instead (built in
# the DBSCAN cell below as DBSCAN_FEATURES_PCA) — kept separate because
# DBSCAN is distance-based and benefits from lower dimensionality, while
# Isolation Forest handles the full feature set fine as-is.
ISO_FEATURES = SYM_COLS + ["DAY_INDEX_NORM", "AGE_AT_VISIT_NORM",
                            "COMORBIDITY_COUNT_NORM", "RECENT_CASE_COUNT_NORM"]

print(f"Isolation Forest features : {len(ISO_FEATURES)}")


Isolation Forest features : 32


## Step 8: Isolation Forest Baseline

In [14]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Contamination: realistic expected outbreak rate (not from the label itself)
# Using a conservative fixed estimate avoids leaking label information
CONTAMINATION = 0.025  # ~2.5% expected outbreak rate in the wild
# Raised from 0.015 after reviewing results — still an independent estimate,
# not read from IS_OUTBREAK, just a closer realistic guess of prevalence

iso = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    random_state=42,
    n_jobs=-1
)

X_iso_train = train_df[ISO_FEATURES].values
X_iso_test  = test_df[ISO_FEATURES].values
y_test       = test_df["IS_OUTBREAK"].values

iso.fit(X_iso_train)

iso_raw    = iso.predict(X_iso_test)
iso_pred   = (iso_raw == -1).astype(int)  # -1 = anomaly = predicted outbreak
iso_scores = -iso.score_samples(X_iso_test)  # higher = more anomalous

print("Isolation Forest results on test set:")
print(f"  Precision : {precision_score(y_test, iso_pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test, iso_pred, zero_division=0):.4f}")
print(f"  F1        : {f1_score(y_test, iso_pred, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, iso_scores):.4f}")
print(f"  FPR       : {iso_pred[y_test==0].sum() / (y_test==0).sum():.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, iso_pred)
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Outbreak"]).plot(
    cmap="Blues", ax=ax)
ax.set_title("Isolation Forest")
plt.tight_layout()
plt.savefig("results/iso_forest_cm.png", dpi=150)
plt.close()
print("Saved: results/iso_forest_cm.png")


Isolation Forest results on test set:
  Precision : 0.1268
  Recall    : 0.1189
  F1        : 0.1228
  ROC-AUC   : 0.7523
  FPR       : 0.0231
Saved: results/iso_forest_cm.png


## Step 9: City-Chunked DBSCAN with Symptom Burstiness Scoring

In [15]:
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

def dbscan_outbreak_score(df_chunk, features, eps=0.5, min_samples=5):
    """
    Run DBSCAN on a single city-disease chunk.
    Returns an anomaly score per row: the mean RECENT_CASE_COUNT (burstiness)
    of the cluster each row belongs to. Noise points score 0.

    IMPORTANT: scores are left as RAW RECENT_CASE_COUNT values, not rescaled
    to [0, 1] per chunk. Per-chunk max-normalization was tried first and
    broke comparability across chunks: a chunk that is entirely background
    would still get rescaled so its "least quiet" cluster hits 1.0, making
    it look exactly as severe as a chunk with a genuine outbreak. Raw counts
    stay on the same absolute scale everywhere, so a single global threshold
    (percentile across the whole test set) is meaningful.
    """
    X = df_chunk[features].values
    if len(X) < min_samples:
        return np.zeros(len(X))

    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)

    scores = np.zeros(len(X))
    unique_labels = set(labels) - {-1}

    for lbl in unique_labels:
        mask = labels == lbl
        cluster_burst = df_chunk.loc[df_chunk.index[mask], "RECENT_CASE_COUNT"].mean()
        scores[mask] = cluster_burst

    return scores


def run_chunked_dbscan(df, features, eps=0.5, min_samples=5):
    """
    Apply city-chunked DBSCAN across all city-disease combinations.
    Memory-safe: processes one chunk at a time.
    """
    scores = np.zeros(len(df))
    df = df.reset_index(drop=True)

    for (city, disease), chunk in df.groupby(["CITY", "DISEASE"]):
        chunk_scores = dbscan_outbreak_score(chunk, features, eps, min_samples)
        scores[chunk.index] = chunk_scores

    return scores


print("Running DBSCAN on validation set for hyperparameter tuning...")
y_val = val_df["IS_OUTBREAK"].values

# Reduce symptom dimensionality with PCA before DBSCAN for speed + quality
pca = PCA(n_components=8, random_state=42)
pca.fit(train_df[SYM_COLS].values)

val_df_pca = val_df.copy()
test_df_pca = test_df.copy()

pca_cols = [f"pca_{i}" for i in range(8)]
val_df_pca[pca_cols]  = pca.transform(val_df[SYM_COLS].values)
test_df_pca[pca_cols] = pca.transform(test_df[SYM_COLS].values)

# PCA symptom components + continuous time + burst signal
DBSCAN_FEATURES_PCA = pca_cols + ["DAY_INDEX_NORM", "RECENT_CASE_COUNT_NORM"]

val_scores = run_chunked_dbscan(val_df_pca, DBSCAN_FEATURES_PCA)
val_auc    = roc_auc_score(y_val, val_scores)
print(f"Validation ROC-AUC (default eps=0.5): {val_auc:.4f}")


Running DBSCAN on validation set for hyperparameter tuning...
Validation ROC-AUC (default eps=0.5): 0.7655


## Step 10: DBSCAN Hyperparameter Tuning on Validation Set

In [16]:
import itertools

eps_values         = [0.3, 0.5, 0.7, 1.0, 1.3, 1.6]
min_samples_values = [3, 5, 10]

best_auc    = 0
best_params = {"eps": 0.5, "min_samples": 5}
results     = []

print("Tuning DBSCAN on validation set...")
for eps, min_s in itertools.product(eps_values, min_samples_values):
    scores = run_chunked_dbscan(val_df_pca, DBSCAN_FEATURES_PCA, eps=eps, min_samples=min_s)
    try:
        auc_val = roc_auc_score(y_val, scores)
    except Exception:
        auc_val = 0.0

    threshold = np.percentile(scores, 100 * (1 - CONTAMINATION))
    preds = (scores >= threshold).astype(int)
    f1_val = f1_score(y_val, preds, zero_division=0)

    results.append({"eps": eps, "min_samples": min_s,
                    "val_auc": round(auc_val, 4), "val_f1": round(f1_val, 4)})

    if auc_val > best_auc:
        best_auc    = auc_val
        best_params = {"eps": eps, "min_samples": min_s}

results_df = pd.DataFrame(results).sort_values("val_auc", ascending=False)
print("Top combinations:")
print(results_df.head(8).to_string(index=False))
print(f"Best params: {best_params} | Best Val AUC: {best_auc:.4f}")

# Sanity check: warn if the best value is sitting at the edge of the tested grid --
# that means the true optimum might be outside the range we searched.
if best_params["eps"] == max(eps_values):
    print("\nWARNING: best eps is at the top of the grid -- consider testing higher values "
          "(e.g. 2.0) before trusting this as the final optimum.")
if best_params["min_samples"] == min(min_samples_values):
    print("WARNING: best min_samples is at the bottom of the grid -- consider testing lower "
          "values before trusting this as the final optimum.")

# Sanity check: flag if many combinations tie on the same score -- this can indicate one
# feature (e.g. RECENT_CASE_COUNT_NORM) is dominating the distance calculation so strongly
# that eps/min_samples barely matter anymore.
top_auc = results_df["val_auc"].iloc[0]
n_tied = (results_df["val_auc"] == top_auc).sum()
if n_tied >= 4:
    print(f"\nNOTE: {n_tied} parameter combinations tied at the same val_auc ({top_auc}). "
          "This can mean one feature is dominating the clustering distance -- worth checking "
          "before reporting these hyperparameters as meaningfully 'tuned'.")


Tuning DBSCAN on validation set...
Top combinations:
 eps  min_samples  val_auc  val_f1
 1.6           10   0.9016  0.4345
 1.3            3   0.9016  0.4345
 1.6            5   0.9016  0.4345
 1.6            3   0.9016  0.4345
 1.3           10   0.9016  0.4345
 1.3            5   0.9016  0.4345
 1.0            3   0.9003  0.4340
 1.0            5   0.9002  0.4339
Best params: {'eps': 1.3, 'min_samples': 3} | Best Val AUC: 0.9016

NOTE: 6 parameter combinations tied at the same val_auc (0.9016). This can mean one feature is dominating the clustering distance -- worth checking before reporting these hyperparameters as meaningfully 'tuned'.


## Step 11: Final DBSCAN Evaluation on Test Set

In [17]:
# Run final DBSCAN with best hyperparameters — test set used only once
dbscan_scores = run_chunked_dbscan(
    test_df_pca, DBSCAN_FEATURES_PCA,
    eps=best_params["eps"], min_samples=best_params["min_samples"]
)

# Convert continuous score to binary prediction using contamination threshold
threshold_db = np.percentile(dbscan_scores, 100 * (1 - CONTAMINATION))
dbscan_pred  = (dbscan_scores >= threshold_db).astype(int)

print("DBSCAN results on test set:")
print(f"  Best params : {best_params}")
print(f"  Precision   : {precision_score(y_test, dbscan_pred, zero_division=0):.4f}")
print(f"  Recall      : {recall_score(y_test, dbscan_pred, zero_division=0):.4f}")
print(f"  F1          : {f1_score(y_test, dbscan_pred, zero_division=0):.4f}")
print(f"  ROC-AUC     : {roc_auc_score(y_test, dbscan_scores):.4f}")
print(f"  FPR         : {dbscan_pred[y_test==0].sum() / (y_test==0).sum():.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, dbscan_pred)
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Outbreak"]).plot(
    cmap="Oranges", ax=ax)
ax.set_title(f"DBSCAN (eps={best_params['eps']}, min_samples={best_params['min_samples']})")
plt.tight_layout()
plt.savefig("results/dbscan_cm.png", dpi=150)
plt.close()
print("Saved: results/dbscan_cm.png")


DBSCAN results on test set:
  Best params : {'eps': 1.3, 'min_samples': 3}
  Precision   : 0.4897
  Recall      : 0.4618
  F1          : 0.4753
  ROC-AUC     : 0.9233
  FPR         : 0.0136
Saved: results/dbscan_cm.png


## Step 11b: Error Analysis — Which Outbreaks Does DBSCAN Miss?

Before trying new features, look at what's actually being missed. If missed outbreaks
cluster around specific cities, diseases, or outbreak sizes, that tells us what a new
feature needs to fix -- rather than guessing.


In [18]:
# Break down false negatives (true outbreak rows the model predicted as normal) by
# city, disease, and outbreak event size, to see if misses follow a pattern.
test_df_eval = test_df.reset_index(drop=True).copy()
test_df_eval["dbscan_pred"] = dbscan_pred
test_df_eval["dbscan_score"] = dbscan_scores

missed = test_df_eval[(test_df_eval["IS_OUTBREAK"] == 1) & (test_df_eval["dbscan_pred"] == 0)]
caught = test_df_eval[(test_df_eval["IS_OUTBREAK"] == 1) & (test_df_eval["dbscan_pred"] == 1)]

print(f"Total true outbreak rows in test set : {len(test_df_eval[test_df_eval['IS_OUTBREAK']==1]):,}")
print(f"Missed (false negatives)             : {len(missed):,} ({len(missed)/max((test_df_eval['IS_OUTBREAK']==1).sum(),1):.1%})")
print(f"Caught (true positives)              : {len(caught):,}")

print("\nMissed outbreak rows by CITY (top 10):")
print(missed["CITY"].value_counts().head(10))

print("\nMissed outbreak rows by DISEASE (top 10):")
print(missed["DISEASE"].value_counts().head(10))

# Outbreak "event size": how many rows share the same CITY + DISEASE + calendar week --
# small events are inherently harder to separate from background noise than large ones.
test_df_eval["_WEEK"] = test_df_eval["VISIT_DATE"].dt.to_period("W").astype(str)
event_size = (test_df_eval[test_df_eval["IS_OUTBREAK"] == 1]
              .groupby(["CITY", "DISEASE", "_WEEK"]).size().rename("event_size"))
test_df_eval = test_df_eval.merge(
    event_size.reset_index(), on=["CITY", "DISEASE", "_WEEK"], how="left"
)

print("\nMean outbreak event size -- missed vs caught:")
print("  Missed:", test_df_eval.loc[(test_df_eval['IS_OUTBREAK']==1) & (test_df_eval['dbscan_pred']==0), 'event_size'].mean().round(1))
print("  Caught:", test_df_eval.loc[(test_df_eval['IS_OUTBREAK']==1) & (test_df_eval['dbscan_pred']==1), 'event_size'].mean().round(1))
print("\nIf missed events are consistently smaller than caught events, the model is likely")
print("under-sensitive to small/early outbreaks -- exactly the case an early-warning system")
print("cares most about catching.")


Total true outbreak rows in test set : 25,421
Missed (false negatives)             : 13,682 (53.8%)
Caught (true positives)              : 11,739

Missed outbreak rows by CITY (top 10):
CITY
Lagos         3686
Alexandria    3551
Abuja         1007
Damietta       978
Tunis          976
Hurghada       879
Casablanca     745
Kumasi         738
Qena           470
Damanhur       272
Name: count, dtype: int64

Missed outbreak rows by DISEASE (top 10):
DISEASE
Lassa fever    3685
Malaria        3438
Influenza      2750
COVID-19       2601
Measles         738
Cholera         470
Name: count, dtype: int64

Mean outbreak event size -- missed vs caught:
  Missed: 413.2
  Caught: 1323.8

If missed events are consistently smaller than caught events, the model is likely
under-sensitive to small/early outbreaks -- exactly the case an early-warning system
cares most about catching.


## Step 11c: Experiment — Population-Normalized Burst Score

`RECENT_CASE_COUNT` is a raw count, not adjusted for city size. A large city's normal
baseline could be higher than a small city's genuine outbreak, so a single global
threshold might favor big cities. This tests a per-(CITY, DISEASE) baseline z-score
instead, computed causally (only using data strictly before each row, no leakage).

**This is a validation-set-only comparison.** It does NOT touch the test set, and does
NOT get applied automatically. Read the printed comparison, and only if the z-score
version is clearly better on validation, manually swap the scoring function and re-run
Steps 10-15 once, end to end, with the test set used only that one final time.


In [19]:
def add_burst_zscore(df, count_col="RECENT_CASE_COUNT", baseline_days=60):
    """
    Per (CITY, DISEASE) baseline z-score of RECENT_CASE_COUNT: how unusual is the
    current burst level relative to that same city-disease's OWN recent history,
    rather than an absolute count compared across all cities equally.

    Causal: baseline mean/std use only a backward-looking rolling window, shifted by
    one row so the current value never contributes to its own baseline.
    """
    df = df.sort_values("VISIT_DATE").reset_index(drop=True)
    z = np.zeros(len(df))
    for (city, disease), grp in df.groupby(["CITY", "DISEASE"], sort=False):
        s = grp.set_index("VISIT_DATE")[count_col]
        baseline_mean = s.rolling(f"{baseline_days}D", min_periods=5).mean().shift(1)
        baseline_std  = s.rolling(f"{baseline_days}D", min_periods=5).std().shift(1).replace(0, np.nan)
        zscore = ((s - baseline_mean) / baseline_std).fillna(0)
        z[grp.index] = zscore.values
    return z


def dbscan_outbreak_score_zscore(df_chunk, features, eps, min_samples, score_col):
    """Same clustering as dbscan_outbreak_score, but the per-cluster score is the mean
    of `score_col` (the z-score feature) instead of the raw RECENT_CASE_COUNT."""
    X = df_chunk[features].values
    if len(X) < min_samples:
        return np.zeros(len(X))
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
    scores = np.zeros(len(X))
    for lbl in set(labels) - {-1}:
        mask = labels == lbl
        scores[mask] = df_chunk.loc[df_chunk.index[mask], score_col].mean()
    return scores


def run_chunked_dbscan_zscore(df, features, score_col, eps, min_samples):
    scores = np.zeros(len(df))
    df = df.reset_index(drop=True)
    for (city, disease), chunk in df.groupby(["CITY", "DISEASE"]):
        scores[chunk.index] = dbscan_outbreak_score_zscore(chunk, features, eps, min_samples, score_col)
    return scores


# Compute the z-score feature on train+val only (this experiment never touches test)
train_val_z_input = pd.concat(
    [train_df.assign(_S="train"), val_df.assign(_S="val")], ignore_index=True
)
train_val_z_input["BURST_ZSCORE"] = add_burst_zscore(train_val_z_input)
val_df_pca_z = val_df_pca.copy()
val_df_pca_z["BURST_ZSCORE"] = (
    train_val_z_input[train_val_z_input["_S"] == "val"]
    .reset_index(drop=True)["BURST_ZSCORE"]
)

zscore_val_scores = run_chunked_dbscan_zscore(
    val_df_pca_z, DBSCAN_FEATURES_PCA, "BURST_ZSCORE",
    eps=best_params["eps"], min_samples=best_params["min_samples"]
)
zscore_val_auc = roc_auc_score(y_val, zscore_val_scores)

zscore_threshold = np.percentile(zscore_val_scores, 100 * (1 - CONTAMINATION))
zscore_val_pred = (zscore_val_scores >= zscore_threshold).astype(int)
zscore_val_f1 = f1_score(y_val, zscore_val_pred, zero_division=0)

print("Validation comparison -- raw count vs population-normalized z-score:")
print(f"  Current (raw RECENT_CASE_COUNT) : val_auc={best_auc:.4f}")
print(f"  New (BURST_ZSCORE)              : val_auc={zscore_val_auc:.4f}, val_f1={zscore_val_f1:.4f}")

if zscore_val_auc > best_auc + 0.01:
    print("\nRESULT: the z-score version is meaningfully better on validation. "
          "Consider swapping the scoring function and re-running Steps 10-15 once, "
          "end to end, using the test set only that one final time.")
else:
    print("\nRESULT: no meaningful improvement from population normalization on this data. "
          "Keep the current raw-count scoring -- simpler, and not worse.")


Validation comparison -- raw count vs population-normalized z-score:
  Current (raw RECENT_CASE_COUNT) : val_auc=0.9016
  New (BURST_ZSCORE)              : val_auc=0.6487, val_f1=0.0678

RESULT: no meaningful improvement from population normalization on this data. Keep the current raw-count scoring -- simpler, and not worse.


## Step 12: Model Comparison — ROC and Precision-Recall Curves

In [20]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
fpr_iso, tpr_iso, _ = roc_curve(y_test, iso_scores)
fpr_db,  tpr_db,  _ = roc_curve(y_test, dbscan_scores)

axes[0].plot(fpr_iso, tpr_iso, color="steelblue", lw=2,
             label=f"Isolation Forest (AUC={roc_auc_score(y_test, iso_scores):.3f})")
axes[0].plot(fpr_db,  tpr_db,  color="darkorange", lw=2,
             label=f"DBSCAN (AUC={roc_auc_score(y_test, dbscan_scores):.3f})")
axes[0].plot([0,1],[0,1],"k--",lw=1,label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

# Precision-Recall curves
prec_iso, rec_iso, _ = precision_recall_curve(y_test, iso_scores)
prec_db,  rec_db,  _ = precision_recall_curve(y_test, dbscan_scores)

axes[1].plot(rec_iso, prec_iso, color="steelblue", lw=2,
             label=f"Isolation Forest (AUC={auc(rec_iso, prec_iso):.3f})")
axes[1].plot(rec_db,  prec_db,  color="darkorange", lw=2,
             label=f"DBSCAN (AUC={auc(rec_db, prec_db):.3f})")
axes[1].axhline(y=y_test.mean(), color="k", linestyle="--", lw=1,
                label=f"Baseline prevalence={y_test.mean():.4f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)

plt.suptitle("EpiSentinel Task 3 — Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/task3_roc_pr_curves.png", dpi=150)
plt.close()
print("Saved: results/task3_roc_pr_curves.png")

# Summary table
summary = pd.DataFrame([
    {
        "Model"    : "Isolation Forest",
        "Precision": round(precision_score(y_test, iso_pred,    zero_division=0), 4),
        "Recall"   : round(recall_score(y_test, iso_pred,       zero_division=0), 4),
        "F1"       : round(f1_score(y_test, iso_pred,           zero_division=0), 4),
        "ROC-AUC"  : round(roc_auc_score(y_test, iso_scores),                    4),
        "FPR"      : round(iso_pred[y_test==0].sum()/(y_test==0).sum(),           4),
    },
    {
        "Model"    : "DBSCAN",
        "Precision": round(precision_score(y_test, dbscan_pred, zero_division=0), 4),
        "Recall"   : round(recall_score(y_test, dbscan_pred,    zero_division=0), 4),
        "F1"       : round(f1_score(y_test, dbscan_pred,        zero_division=0), 4),
        "ROC-AUC"  : round(roc_auc_score(y_test, dbscan_scores),                 4),
        "FPR"      : round(dbscan_pred[y_test==0].sum()/(y_test==0).sum(),        4),
    },
])
print("Model comparison:")
print(summary.to_string(index=False))
summary.to_csv("results/task3_model_comparison.csv", index=False)
print("Saved: results/task3_model_comparison.csv")


Saved: results/task3_roc_pr_curves.png
Model comparison:
           Model  Precision  Recall     F1  ROC-AUC    FPR
Isolation Forest     0.1268  0.1189 0.1228   0.7523 0.0231
          DBSCAN     0.4897  0.4618 0.4753   0.9233 0.0136
Saved: results/task3_model_comparison.csv


## Step 13: Early Warning Lead Time Analysis

In [21]:
# Measure how many days before an outbreak the model raises a SUSTAINED alert,
# within the same city-disease series. Lead time is computed in calendar days.
#
# FIX: the original version credited "early warning" to any single flagged row
# anywhere in the lookback window, even an isolated false positive. With a non-trivial
# background false-positive rate, at least one row being flagged somewhere in a 42-day
# window (across many rows per city-disease group) happens often by pure chance --
# Monte Carlo testing showed this credits "early warning" on PURE NOISE 57-100% of the
# time depending on the background FPR. Requiring a SUSTAINED run of alert-days (not a
# single stray row) fixes this: a real building signal shows up as several nearby
# alert days, not one random blip.

MAX_LOOKBACK_DAYS = 42       # 6 weeks
MIN_SUSTAINED_ALERTS = 3     # minimum alert-days required
SUSTAINED_WINDOW_DAYS = 3    # ...within this many consecutive calendar days

def compute_lead_times_days(df, predictions, max_days=MAX_LOOKBACK_DAYS,
                             min_sustained=MIN_SUSTAINED_ALERTS,
                             sustained_window=SUSTAINED_WINDOW_DAYS):
    df = df.copy().reset_index(drop=True)
    df["pred"] = predictions
    df["VISIT_DATE"] = pd.to_datetime(df["VISIT_DATE"])
    df = df.sort_values("VISIT_DATE").reset_index(drop=True)

    outbreak_idx = df.index[df["IS_OUTBREAK"] == 1].tolist()
    lead_days = []

    for idx in outbreak_idx:
        outbreak_date = df.at[idx, "VISIT_DATE"]
        city          = df.at[idx, "CITY"]
        disease       = df.at[idx, "DISEASE"]

        window = df[
            (df["CITY"] == city) &
            (df["DISEASE"] == disease) &
            (df["VISIT_DATE"] < outbreak_date) &
            (df["VISIT_DATE"] >= outbreak_date - pd.Timedelta(days=max_days))
        ]

        if len(window) == 0:
            lead_days.append(0)
            continue

        daily = window.groupby(window["VISIT_DATE"].dt.floor("D"))["pred"].max()
        daily = daily.reindex(
            pd.date_range(daily.index.min(), daily.index.max(), freq="D"), fill_value=0
        )
        rolling = daily.rolling(sustained_window, min_periods=1).sum()
        sustained_days = rolling[rolling >= min_sustained]

        if len(sustained_days) > 0:
            first_alert_date = sustained_days.index.min()
            lead_days.append(max((outbreak_date - first_alert_date).days, 0))
        else:
            lead_days.append(0)  # no sustained early warning -- missed or too late

    return lead_days

lead_iso_days    = compute_lead_times_days(test_df, iso_pred)
lead_dbscan_days = compute_lead_times_days(test_df, dbscan_pred)

n_events = len(lead_iso_days)
print(f"Outbreak events in test set: {n_events}")
print()
print("Isolation Forest early warning (sustained-alert method):")
print(f"  Avg lead time : {np.mean(lead_iso_days):.1f} days")
print(f"  Max lead time : {max(lead_iso_days)} days")
print(f"  Events with early warning: {sum(l > 0 for l in lead_iso_days)} / {n_events}")

print()
print("DBSCAN early warning (sustained-alert method):")
print(f"  Avg lead time : {np.mean(lead_dbscan_days):.1f} days")
print(f"  Max lead time : {max(lead_dbscan_days)} days")
print(f"  Events with early warning: {sum(l > 0 for l in lead_dbscan_days)} / {n_events}")


Outbreak events in test set: 25421

Isolation Forest early warning (sustained-alert method):
  Avg lead time : 20.6 days
  Max lead time : 40 days
  Events with early warning: 22501 / 25421

DBSCAN early warning (sustained-alert method):
  Avg lead time : 18.4 days
  Max lead time : 40 days
  Events with early warning: 11740 / 25421


## Step 14: Lead Time Visualization

In [22]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins = range(0, MAX_LOOKBACK_DAYS + 8, 7)  # weekly bins

axes[0].hist(lead_iso_days, bins=bins, align="left",
             color="steelblue", edgecolor="white", rwidth=0.8)
axes[0].set_xlabel("Lead Time (days before outbreak)")
axes[0].set_ylabel("Number of outbreak events")
axes[0].set_title(f"Isolation Forest\nAvg lead: {np.mean(lead_iso_days):.1f} days")
axes[0].grid(axis="y", alpha=0.3)

axes[1].hist(lead_dbscan_days, bins=bins, align="left",
             color="darkorange", edgecolor="white", rwidth=0.8)
axes[1].set_xlabel("Lead Time (days before outbreak)")
axes[1].set_ylabel("Number of outbreak events")
axes[1].set_title(f"DBSCAN\nAvg lead: {np.mean(lead_dbscan_days):.1f} days")
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("EpiSentinel Task 3 — Early Warning Lead Time", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/task3_lead_time.png", dpi=150)
plt.close()
print("Saved: results/task3_lead_time.png")


Saved: results/task3_lead_time.png


## Step 15: Save Models and Metadata for Task 4

In [23]:
import pickle, json

# Save models — artifacts/ (needed to reload/resume training, not final results)
pickle.dump(iso,     open("artifacts/task3_iso_forest.pkl", "wb"))
pickle.dump(pca,     open("artifacts/task3_pca.pkl",         "wb"))
pickle.dump(best_params, open("artifacts/task3_dbscan_params.pkl", "wb"))

# Metadata for Task 4 — results/
metadata = {
    "models": {
        "IsolationForest": {
            "file"           : "artifacts/task3_iso_forest.pkl",
            "features"       : ISO_FEATURES,
            "contamination"  : CONTAMINATION,
        },
        "DBSCAN": {
            "params_file"    : "artifacts/task3_dbscan_params.pkl",
            "pca_file"       : "artifacts/task3_pca.pkl",
            "pca_components" : 8,
            "features_after_pca": DBSCAN_FEATURES_PCA,
            "threshold"      : float(threshold_db),
        },
        "RecentCaseRate": {
            "scaler_file"    : "artifacts/case_rate_scaler.pkl",
            "window_days"    : CASE_RATE_WINDOW_DAYS,
        },
    },
    "test_metrics": {
        "IsolationForest": {
            "precision": round(precision_score(y_test, iso_pred,    zero_division=0), 4),
            "recall"   : round(recall_score(y_test, iso_pred,       zero_division=0), 4),
            "f1"       : round(f1_score(y_test, iso_pred,           zero_division=0), 4),
            "roc_auc"  : round(roc_auc_score(y_test, iso_scores),                    4),
            "fpr"      : round(iso_pred[y_test==0].sum()/(y_test==0).sum(),           4),
        },
        "DBSCAN": {
            "precision": round(precision_score(y_test, dbscan_pred, zero_division=0), 4),
            "recall"   : round(recall_score(y_test, dbscan_pred,    zero_division=0), 4),
            "f1"       : round(f1_score(y_test, dbscan_pred,        zero_division=0), 4),
            "roc_auc"  : round(roc_auc_score(y_test, dbscan_scores),                 4),
            "fpr"      : round(dbscan_pred[y_test==0].sum()/(y_test==0).sum(),        4),
        },
    },
    "early_warning": {
        "IsolationForest": {
            "avg_lead_days"         : round(float(np.mean(lead_iso_days)), 1),
            "max_lead_days"         : int(max(lead_iso_days)),
            "events_with_warning"   : int(sum(l > 0 for l in lead_iso_days)),
            "total_events"          : n_events,
        },
        "DBSCAN": {
            "avg_lead_days"         : round(float(np.mean(lead_dbscan_days)), 1),
            "max_lead_days"         : int(max(lead_dbscan_days)),
            "events_with_warning"   : int(sum(l > 0 for l in lead_dbscan_days)),
            "total_events"          : n_events,
        },
    },
    "output_files": {
        # artifacts/ — everything needed to reload or resume the pipeline
        # (encoders, scaler, split data, trained model objects). Not the
        # final answer on its own — just what training depends on.
        "artifacts": [
            "artifacts/encoders.pkl",
            "artifacts/scaler.pkl",
            "artifacts/case_rate_scaler.pkl",
            "artifacts/train_data.csv",
            "artifacts/val_data.csv",
            "artifacts/test_data.csv",
            "artifacts/task3_iso_forest.pkl",
            "artifacts/task3_pca.pkl",
            "artifacts/task3_dbscan_params.pkl",
        ],
        # results/ — final numbers and final visuals, ready to present
        "results": [
            "results/task3_metadata.json",
            "results/task3_model_comparison.csv",
            "results/iso_forest_cm.png",
            "results/dbscan_cm.png",
            "results/task3_roc_pr_curves.png",
            "results/task3_lead_time.png",
        ],
    },
}

with open("results/task3_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("All Task 3 files saved:")
for folder, files in metadata["output_files"].items():
    print(f"  {folder}/")
    for fname in files:
        print(f"    {fname}")


All Task 3 files saved:
  artifacts/
    artifacts/encoders.pkl
    artifacts/scaler.pkl
    artifacts/case_rate_scaler.pkl
    artifacts/train_data.csv
    artifacts/val_data.csv
    artifacts/test_data.csv
    artifacts/task3_iso_forest.pkl
    artifacts/task3_pca.pkl
    artifacts/task3_dbscan_params.pkl
  results/
    results/task3_metadata.json
    results/task3_model_comparison.csv
    results/iso_forest_cm.png
    results/dbscan_cm.png
    results/task3_roc_pr_curves.png
    results/task3_lead_time.png
